# Practice: Reading ADCP Data with pycurrents

NEEDS WORKS

This notebook walks through reading the demo cruise data 

Launched Jupyter from inside the `pycodas` conda environment:

```bash
conda activate pycodas
jupyter notebook
```

**Data location:** `~/adcpcode/programs/codas_demos/uhdas_data/

---

## 1. Imports

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# from pycurrents import hints
from pycurrents.codas import get_profiles
from pycurrents.system import Bunch


## Example of tools
[Link to Eric's notes](https://currents.soest.hawaii.edu/ocn_data_analysis/installation.html#quick-test)
*mapper* --> quick map example below

In [2]:
%matplotlib qt
from pycurrents.plot.maptools import mapper
m = mapper([-163, -153], [18, 25])
m.topo()
m.grid()

/Users/alesan/miniforge3/envs/pycodas/lib/python3.13/site-packages/matplotlib/colors.py:778: RuntimeWarning: overflow encountered in multiply
  xa *= self.N


---
## 2. Point to the demo dataset

The CODAS binary database lives in the `adcpdb/` subfolder.  
`a_km` is the database stem — pycurrents finds the `.blk`, `.bbn`, etc. files automatically.

In [3]:
cruise_dir = Path.home() / 'adcpcode/programs/codas_demos/uhdas_data/km1001c/proc/os38nb'
db_path = cruise_dir / 'adcpdb' / 'a_km'


---
## 3. Read the data

`get_profiles()` reads the CODAS binary database and returns a `Bunch` (a dict-like object).  
Key variables inside:
- `data.umeas`, `data.vmeas` — measured eastward / northward **water** velocity (m/s)
- `data.depth` — bin depths (m)
- `data.dday` — time as decimal day-of-year
- `data.lon`, `data.lat` — ship position
- `data.amp` — acoustic backscatter amplitude (counts)
- `data.pg` — percent good (data quality, 0–100)

In [4]:
data = get_profiles(str(db_path))

print(type(data))
print('Variables available:', list(data.keys()))

<class 'pycurrents.codas.tools.ProcEns'>
Variables available: ['nprofs', 'nbins', 'names', 'blkprf', 'ymdhms', 'lgb', 'mab', 'pflag', 'pg', 'amp', 'swcor', 'ra', 'depth', 'umeas', 'vmeas', 'w', 'e', 'uship', 'vship', 'heading', 'tr_temp', 'snd_spd_used', 'best_snd_spd', 'watrk_hd_misalign', 'watrk_scale_factor', 'botrk_hd_misalign', 'botrk_scale_factor', 'last_temp', 'last_heading', 'mn_pitch', 'mn_roll', 'std_pitch', 'std_roll', 'u_bt', 'v_bt', 'd_bt', 'dday', 'lon_raw', 'lat_raw', 'lon', 'lat', 'resid_stats', 'tseries_stats', 'tseries_diffstats', 'pgs_sample', 'num_bins', 'e_std', 'yearbase', 'dep', 'bins', 'amp1', 'amp2', 'amp3', 'amp4', 'use_bt_for_shipspeed', 'velmask', 'navmask', 'newmask', 'u', 'v', 'configs', 'config_start_indices', 'CellSize', 'Pulse', 'depth_interval', 'Bin1Dist']


# KM dataset

---
## 4. Inspect shapes and basic stats

The velocity arrays are 2-D: `(time_steps, depth_bins)`.

In [5]:
print('u shape (time x depth):', data.umeas.shape)
print('depth bins:', data.depth)
print('Time range (dday):', data.dday.min(), '-->', data.dday.max())
print('Position range — lon:', data.lon.min(), data.lon.max())
print('Position range — lat:', data.lat.min(), data.lat.max())

u shape (time x depth): (532, 70)
depth bins: [[  46.96   70.96   94.96 ... 1654.96 1678.96 1702.96]
 [  46.96   70.96   94.96 ... 1654.96 1678.96 1702.96]
 [  46.96   70.96   94.96 ... 1654.96 1678.96 1702.96]
 ...
 [  47.     71.     95.   ... 1655.   1679.   1703.  ]
 [  47.     71.     95.   ... 1655.   1679.   1703.  ]
 [  47.     71.     95.   ... 1655.   1679.   1703.  ]]
Time range (dday): 29.74054398148148 --> 31.6571875
Position range — lon: -125.92130555555556 -124.39970555555556
Position range — lat: 43.383266666666664 44.503908333333335


---
## 5. Quick velocity section plot

Plot eastward velocity (u) as a time-depth section — the classic ADCP view.

In [7]:
print('dday shape:', data.dday.shape)
print('depth shape:', data.depth.shape)
print('umeas shape:', data.umeas.shape)


dday shape: (532,)
depth shape: (532, 70)
umeas shape: (532, 70)


In [9]:
depth_1d = data.depth.mean(axis=0)   # shape (70,)

fig, ax = plt.subplots(figsize=(12, 4))

pcm = ax.pcolormesh(
    data.dday,
    depth_1d,
    data.umeas.T,        # transpose so depth is on the y-axis
    cmap='RdBu_r',
    vmin=-2.0, vmax=2.0
)

ax.invert_yaxis()       # depth increases downward
ax.set_xlabel('Decimal day')
ax.set_ylabel('Depth (m)')
ax.set_title('Eastward velocity — km1001c os38nb')
plt.colorbar(pcm, ax=ax, label='u (m/s)')
plt.tight_layout()
plt.show()

---
## 6. Plot backscatter amplitude

Amplitude (echo intensity) shows signal strength. High values near the surface, drops off with depth.  
Spikes can indicate biology (sound-scattering layers).

In [11]:
fig, ax = plt.subplots(figsize=(12, 4))

# amp can be 3-D (time, depth, beam) — take the mean across beams if so
amp = data.amp
if amp.ndim == 3:
    amp = amp.mean(axis=2)

pcm = ax.pcolormesh(
    data.dday,
    depth_1d,
    amp.T,
    cmap='viridis'
)

ax.invert_yaxis()
ax.set_xlabel('Decimal day')
ax.set_ylabel('Depth (m)')
ax.set_title('Backscatter amplitude — km1001c os38nb')
plt.colorbar(pcm, ax=ax, label='Amplitude (counts)')
plt.tight_layout()
plt.show()

---
## 7. Percent good (data quality)

`pg` (percent good) is the fraction of pings that passed quality screening (0–100).  
Values below ~25 are usually masked out in final products.

In [12]:
fig, ax = plt.subplots(figsize=(12, 4))

pg = data.pg
if pg.ndim == 3:
    pg = pg.mean(axis=2)   # average across beams

pcm = ax.pcolormesh(
    data.dday,
    depth_1d,
    pg.T,
    cmap='plasma',
    vmin=0, vmax=100
)

ax.invert_yaxis()
ax.set_xlabel('Decimal day')
ax.set_ylabel('Depth (m)')
ax.set_title('Percent good — km1001c os38nb')
plt.colorbar(pcm, ax=ax, label='% good')
plt.tight_layout()
plt.show()

---
## 8. Single velocity profile at one time step

Pick a specific time index and plot u and v vs depth — a classic "curtain" profile.

In [13]:
time_idx = 50   # change this to explore different profiles

fig, ax = plt.subplots(figsize=(4, 6))

ax.plot(data.uabs[time_idx, :], data.depth, label='u (east)', color='steelblue')
ax.plot(data.vabs[time_idx, :], data.depth, label='v (north)', color='tomato')
ax.axvline(0, color='k', linewidth=0.5)
ax.invert_yaxis()
ax.set_xlabel('Velocity (m/s)')
ax.set_ylabel('Depth (m)')
ax.set_title(f'Profile at time index {time_idx}\n(dday = {data.dday[time_idx]:.3f})')
ax.legend()
plt.tight_layout()
plt.show()

AttributeError: 'ProcEns' object has no attribute 'uabs'

---
## 9. Ship track (lat/lon)

Where was the ship during this cruise segment?

In [14]:
fig, ax = plt.subplots(figsize=(6, 6))

sc = ax.scatter(data.lon, data.lat, c=data.dday, cmap='viridis', s=2)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Ship track — km1001c')
plt.colorbar(sc, ax=ax, label='Decimal day')
plt.tight_layout()
plt.show()

---
## 10. Apply a simple quality mask

Mask out velocity values where percent good is below 25 — common threshold in ADCP processing.

In [15]:
pg2d = data.pg if data.pg.ndim == 2 else data.pg.mean(axis=2)

u_masked = np.where(pg2d >= 25, data.uabs, np.nan)
v_masked = np.where(pg2d >= 25, data.vabs, np.nan)

print('Fraction of u values masked out:', np.isnan(u_masked).mean())

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

for ax, vel, label in zip(axes, [u_masked, v_masked], ['u (east)', 'v (north)']):
    pcm = ax.pcolormesh(data.dday, data.depth, vel.T, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.invert_yaxis()
    ax.set_ylabel('Depth (m)')
    ax.set_title(f'{label} — quality-masked (pg ≥ 25)')
    plt.colorbar(pcm, ax=ax, label='m/s')

axes[-1].set_xlabel('Decimal day')
plt.tight_layout()
plt.show()

AttributeError: 'ProcEns' object has no attribute 'uabs'